In [1]:
import torch
import torch.nn as nn
import math


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # (1, max_len, d_model) 方便广播
        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        output = x + self.pe[:, : x.size(1)]
        print("\n=== 位置编码输出 ===")
        print(f"形状: {output.shape}")
        print("位置编码后的第一个token:", output[0, 0, :3].detach().numpy())
        return output


In [3]:
# ===================== 测试代码 =====================
def test_positional_encoding():
    torch.manual_seed(42)

    batch_size = 2
    seq_len = 5
    d_model = 6

    # 模拟输入 (全0，方便观察位置编码本身)
    x = torch.zeros(batch_size, seq_len, d_model)

    print("=== 输入 ===")
    print(f"形状: {x.shape}")
    print("第一个token:", x[0, 0])

    pe = PositionalEncoding(d_model)

    output = pe(x)

    print("\n=== 验证 ===")

    # 1. shape 检查
    print("shape是否一致:", output.shape == x.shape)

    # 2. 不同位置是否不同
    print("pos0 vs pos1 是否不同:", not torch.allclose(output[0, 0], output[0, 1]))

    # 3. 同一位置在不同 batch 是否一样
    print(
        "batch0 pos0 vs batch1 pos0 是否一样:",
        torch.allclose(output[0, 0], output[1, 0]),
    )

    # 4. 打印前两个位置编码
    print("\n位置0:", output[0, 0])
    print("位置1:", output[0, 1])


test_positional_encoding()

=== 输入 ===
形状: torch.Size([2, 5, 6])
第一个token: tensor([0., 0., 0., 0., 0., 0.])

=== 位置编码输出 ===
形状: torch.Size([2, 5, 6])
位置编码后的第一个token: [0. 1. 0.]

=== 验证 ===
shape是否一致: True
pos0 vs pos1 是否不同: True
batch0 pos0 vs batch1 pos0 是否一样: True

位置0: tensor([0., 1., 0., 1., 0., 1.])
位置1: tensor([0.8415, 0.5403, 0.0464, 0.9989, 0.0022, 1.0000])
